In [1]:
# import os
# os.environ["OPENAI_API_KEY"] = "your-key-here"


  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pandas-2.2.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached regex-2024.11.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (40 kB)
  Using cached ujson-5.10.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.3 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.5.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached multiprocess-0.70.16-py312-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2024.12.0-py3-none-any.whl.metadata (11 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
INFO: pip is looking at multiple versions of litellm to determine w

In [ ]:
import dspy

In [ ]:
class DDLToSQLSignature(dspy.Signature):
    """Generate SQL from natural language question and DDL."""
    question = dspy.InputField(desc="User's natural language question")
    ddl = dspy.InputField(desc="The database schema in DDL format (CREATE TABLE ...)")
    sql = dspy.OutputField(desc="Generated SQL query")

In [ ]:
class DDLToSQLAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(DDLToSQLSignature)

    def forward(self, question, ddl):
        return self.predictor(question=question, ddl=ddl)

In [ ]:
dspy.settings.configure(lm=dspy.OpenAI(model='gpt-4'))

ddl = """
CREATE TABLE customers (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    revenue DECIMAL(10,2),
    region VARCHAR(50)
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    amount DECIMAL(10,2),
    order_date DATE,
    FOREIGN KEY (customer_id) REFERENCES customers(id)
);
"""

In [ ]:
agent = DDLToSQLAgent()

output = agent("Which region has the highest total revenue?", ddl=ddl)
print(output.sql)


In [ ]:
from dspy.teleprompt import BootstrapFewShot

train_examples = [
    dspy.Example(
        question="Show the customer with the highest revenue",
        ddl=ddl,
        sql="SELECT name FROM customers ORDER BY revenue DESC LIMIT 1"
    ).with_inputs("question", "ddl")
]

trained_agent = BootstrapFewShot(DDLToSQLAgent(), trainset=train_examples).compile()

result = trained_agent("List all orders from the customer with the highest revenue", ddl=ddl)
print(result.sql)